# Task 1 (TensorFlow): Adapt the **Acne CNN** notebook to **STL-10**

This notebook adapts the original *acne-classification-using-cnn* workflow to the **STL-10** dataset:
- loads STL-10 (96×96 RGB) using `tensorflow_datasets`
- updates the final layer to **10 classes**
- uses **(Sparse) Categorical Cross-Entropy**
- normalizes inputs consistently
- trains on `train` split and evaluates on `test` split

> Run top-to-bottom. If you are on CPU, start with fewer epochs.

In [6]:
# Download & extract STL-10 (safe, idempotent). Run this cell before reading train_X.bin
import os, sys, tarfile
from urllib.request import urlretrieve

DATA_DIR = './data'
DATA_URL = 'http://ai.stanford.edu/~acoates/stl10/stl10_binary.tar.gz'
ARCHIVE = os.path.join(DATA_DIR, os.path.basename(DATA_URL))
EXTRACTED_DIR = os.path.join(DATA_DIR, 'stl10_binary')
TRAIN_X = os.path.join(EXTRACTED_DIR, 'train_X.bin')
TRAIN_Y = os.path.join(EXTRACTED_DIR, 'train_y.bin')

os.makedirs(DATA_DIR, exist_ok=True)

def _progress(count, block_size, total_size):
    pct = float(count * block_size) / float(total_size) * 100.0
    sys.stdout.write(f'\rDownloading {os.path.basename(ARCHIVE)} {pct:5.1f}%')
    sys.stdout.flush()

if not os.path.exists(EXTRACTED_DIR):
    if not os.path.exists(ARCHIVE):
        print("Downloading STL-10 archive...")
        urlretrieve(DATA_URL, ARCHIVE, reporthook=_progress)
        print("\nDownload complete.")
    print("Extracting archive...")
    with tarfile.open(ARCHIVE, 'r:gz') as t:
        t.extractall(DATA_DIR)
    print("Extraction complete.")

# verify files exist
if not (os.path.exists(TRAIN_X) and os.path.exists(TRAIN_Y)):
    raise FileNotFoundError(f"Expected files not found after extraction: {TRAIN_X}, {TRAIN_Y}")

print("Files ready:", TRAIN_X, TRAIN_Y)

# Now you can call your existing read functions — ensure you open files in binary mode ('rb')
# Example quick read to verify shapes:
import numpy as np

def read_labels(path_to_labels):
    with open(path_to_labels, 'rb') as f:
        return np.fromfile(f, dtype=np.uint8)

def read_all_images(path_to_data):
    with open(path_to_data, 'rb') as f:
        everything = np.fromfile(f, dtype=np.uint8)
    images = np.reshape(everything, (-1, 3, 96, 96))
    images = np.transpose(images, (0, 3, 2, 1))
    return images

labels = read_labels(TRAIN_Y)
images = read_all_images(TRAIN_X)
print("labels.shape:", labels.shape)
print("images.shape:", images.shape)

# Alternative fallback: if network blocked, load via TFDS (will download automatically)
# import tensorflow_datasets as tfds
# (ds_train, ds_test), ds_info = tfds.load("stl10", split=["train","test"], as_supervised=True,

Extracting archive...


/var/folders/0x/6y3p7mjj3g3b3b8wr0t9l2j40000gp/T/ipykernel_88637/2229593937.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(DATA_DIR)


EOFError: Compressed file ended before the end-of-stream marker was reached

In [ ]:
# from __future__ import print_function
# #
# import sys
# import os, sys, tarfile, errno
# import numpy as np
# import matplotlib.pyplot as plt
    
# if sys.version_info >= (3, 0, 0):
#     import urllib.request as urllib # ugly but works
# else:
#     import urllib

# try:
#     from imageio import imsave
# except:
#     from scipy.misc import imsave

# print(sys.version_info) 

# # image shape
# HEIGHT = 96
# WIDTH = 96
# DEPTH = 3

# # size of a single image in bytes
# SIZE = HEIGHT * WIDTH * DEPTH

# # path to the directory with the data
# DATA_DIR = './data'

# # url of the binary data
# DATA_URL = 'http://ai.stanford.edu/~acoates/stl10/stl10_binary.tar.gz'

# # path to the binary train file with image data
# DATA_PATH = './data/stl10_binary/train_X.bin'

# # path to the binary train file with labels
# LABEL_PATH = './data/stl10_binary/train_y.bin'

# def read_labels(path_to_labels):
#     """
#     :param path_to_labels: path to the binary file containing labels from the STL-10 dataset
#     :return: an array containing the labels
#     """
#     with open(path_to_labels, 'rb') as f:
#         labels = np.fromfile(f, dtype=np.uint8)
#         return labels


# def read_all_images(path_to_data):
#     """
#     :param path_to_data: the file containing the binary images from the STL-10 dataset
#     :return: an array containing all the images
#     """

#     with open(path_to_data, 'rb') as f:
#         # read whole file in uint8 chunks
#         everything = np.fromfile(f, dtype=np.uint8)

#         # We force the data into 3x96x96 chunks, since the
#         # images are stored in "column-major order", meaning
#         # that "the first 96*96 values are the red channel,
#         # the next 96*96 are green, and the last are blue."
#         # The -1 is since the size of the pictures depends
#         # on the input file, and this way numpy determines
#         # the size on its own.

#         images = np.reshape(everything, (-1, 3, 96, 96))

#         # Now transpose the images into a standard image format
#         # readable by, for example, matplotlib.imshow
#         # You might want to comment this line or reverse the shuffle
#         # if you will use a learning algorithm like CNN, since they like
#         # their channels separated.
#         images = np.transpose(images, (0, 3, 2, 1))
#         return images


# def read_single_image(image_file):
#     """
#     CAREFUL! - this method uses a file as input instead of the path - so the
#     position of the reader will be remembered outside of context of this method.
#     :param image_file: the open file containing the images
#     :return: a single image
#     """
#     # read a single image, count determines the number of uint8's to read
#     image = np.fromfile(image_file, dtype=np.uint8, count=SIZE)
#     # force into image matrix
#     image = np.reshape(image, (3, 96, 96))
#     # transpose to standard format
#     # You might want to comment this line or reverse the shuffle
#     # if you will use a learning algorithm like CNN, since they like
#     # their channels separated.
#     image = np.transpose(image, (2, 1, 0))
#     return image


# def plot_image(image):
#     """
#     :param image: the image to be plotted in a 3-D matrix format
#     :return: None
#     """
#     plt.imshow(image)
#     plt.show()

# def save_image(image, name):
#     imsave("%s.png" % name, image, format="png")

# def download_and_extract():
#     """
#     Download and extract the STL-10 dataset
#     :return: None
#     """
#     dest_directory = DATA_DIR
#     if not os.path.exists(dest_directory):
#         os.makedirs(dest_directory)
#     filename = DATA_URL.split('/')[-1]
#     filepath = os.path.join(dest_directory, filename)
#     if not os.path.exists(filepath):
#         def _progress(count, block_size, total_size):
#             sys.stdout.write('\rDownloading %s %.2f%%' % (filename,
#                 float(count * block_size) / float(total_size) * 100.0))
#             sys.stdout.flush()
#         filepath, _ = urllib.urlretrieve(DATA_URL, filepath, reporthook=_progress)
#         print('Downloaded', filename)
#         tarfile.open(filepath, 'r:gz').extractall(dest_directory)

# def save_images(images, labels):
#     print("Saving images to disk")
#     i = 0
#     for image in images:
#         label = labels[i]
#         directory = './img/' + str(label) + '/'
#         try:
#             os.makedirs(directory, exist_ok=True)
#         except OSError as exc:
#             if exc.errno == errno.EEXIST:
#                 pass
#         filename = directory + str(i)
#         print(filename)
#         save_image(image, filename)
#         i = i+1
    
# if __name__ == "__main__":
#     # download data if needed
#     download_and_extract()

#     # test to check if the image is read correctly
#     with open(DATA_PATH) as f:
#         image = read_single_image(f)
#         plot_image(image)

#     # test to check if the whole dataset is read correctly
#     images = read_all_images(DATA_PATH)
#     print(images.shape)

#     labels = read_labels(LABEL_PATH)
#     print(labels.shape)

#     # save images to disk
#     save_images(images, labels)

sys.version_info(major=3, minor=13, micro=6, releaselevel='final', serial=0)


FileNotFoundError: [Errno 2] No such file or directory: './data/stl10_binary/train_X.bin'

In [1]:
# ----------------------------
# 0) Imports
# ----------------------------
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# If this import fails, install in your environment:
# !pip install tensorflow-datasets
import tensorflow_datasets as tfds

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, Rescaling
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)

/Users/test/Downloads/Python test/.venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


ModuleNotFoundError: No module named 'tensorflow_datasets'

## 1) Load STL-10 with TFDS (96×96×3)

In [ ]:
# ----------------------------
# PARAMETERS (feel free to tweak)
# ----------------------------
BATCH_SIZE = 64
IMAGE_SIZE = 96
EPOCHS = 10
LEARNING_RATE = 1e-3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ----------------------------
# LOAD DATASETS (STL-10)
# ----------------------------
# tfds returns dicts like {"image": ..., "label": ...}
(ds_train, ds_test), ds_info = tfds.load(
    "stl10",
    split=["train", "test"],
    as_supervised=True,          # returns (image, label)
    with_info=True
)

NUM_CLASSES = ds_info.features["label"].num_classes
CLASS_NAMES = ds_info.features["label"].names

print("Num classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)

# Quick shape check
for x, y in ds_train.take(1):
    print("One sample image shape:", x.shape, "label:", y.numpy())

## 2) Build input pipeline (shuffle, batch, normalize)

In [ ]:
# ----------------------------
# Preprocessing layers (similar spirit to acne notebook)
# ----------------------------
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.05),
    RandomZoom(0.1),
], name="data_augmentation")

normalizer = Rescaling(1./255, name="rescale_01")

AUTOTUNE = tf.data.AUTOTUNE

def preprocess_train(image, label):
    # image is uint8 [0,255], shape (96,96,3)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image, training=True)
    image = normalizer(image)
    return image, label

def preprocess_eval(image, label):
    image = tf.cast(image, tf.float32)
    image = normalizer(image)
    return image, label

train_ds = (ds_train
            .shuffle(5000, seed=SEED, reshuffle_each_iteration=True)
            .map(preprocess_train, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

test_ds  = (ds_test
            .map(preprocess_eval, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

print(train_ds, test_ds)

## 3) Visualize a few samples

In [ ]:
# Visualize a small batch
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 8))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        # images are already normalized float32
        plt.imshow(np.clip(images[i].numpy(), 0, 1))
        plt.title(CLASS_NAMES[int(labels[i].numpy())])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 4) Model (Acne CNN-style) updated for STL-10 (10 classes)

In [ ]:
# ----------------------------
# Model architecture (keeps the acne notebook's Conv/Pool stack vibe)
# - Updated input shape to (96,96,3)
# - Updated final Dense to 10 + softmax
# ----------------------------
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(NUM_CLASSES, activation='softmax')  # <- 10 classes for STL-10
])

model.summary()

## 5) Compile (Sparse Categorical Cross-Entropy)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

## 6) Train & Evaluate (report train + test accuracy)

In [ ]:
# ----------------------------
# Callbacks (same idea as acne notebook)
# ----------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)

history = model.fit(
    train_ds,
    validation_data=test_ds,   # for Task 1, train on train split, evaluate on test split
    epochs=EPOCHS,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

# Final evaluation on test split
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
train_acc = history.history["accuracy"][-1]
print(f"Final train accuracy (last epoch): {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## 7) Overfitting / underfitting quick check

In [ ]:
# Plot accuracy curves
plt.figure()
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="test_acc (as val)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

gap = history.history["accuracy"][-1] - history.history["val_accuracy"][-1]
print("Accuracy gap (train - test):", float(gap))
print("Rule of thumb: a large positive gap suggests overfitting.")